In [1]:
import numpy as np

from Util.Problems import Problem, solution

import math

class P026(Problem):
    number = 26
    title = "Reciprocal Cycles"
    description = """<p>A unit fraction contains $1$ in the numerator. The decimal representation of the unit fractions with denominators $2$ to $10$ are given:</p>
$$\\begin{align}
1/2 &= 0.5\\\\
1/3 &=0.(3)\\\\
1/4 &=0.25\\\\
1/5 &= 0.2\\\\
1/6 &= 0.1(6)\\\\
1/7 &= 0.(142857)\\\\
1/8 &= 0.125\\\\
1/9 &= 0.(1)\\\\
1/10 &= 0.1
\\end{align}$$
<p>Where $0.1(6)$ means $0.166666\\cdots$, and has a $1$-digit recurring cycle. It can be seen that $1/7$ has a $6$-digit recurring cycle.</p><p>Find the value of $d \\lt 1000$ for which $1/d$ contains the longest recurring cycle in its decimal fraction part.</p>"""
    upper_limit = 1000

In [2]:
p = P026()
p.describe()

## Problem 26: Reciprocal Cycles

<p>A unit fraction contains $1$ in the numerator. The decimal representation of the unit fractions with denominators $2$ to $10$ are given:</p>
$$\begin{align}
1/2 &= 0.5\\
1/3 &=0.(3)\\
1/4 &=0.25\\
1/5 &= 0.2\\
1/6 &= 0.1(6)\\
1/7 &= 0.(142857)\\
1/8 &= 0.125\\
1/9 &= 0.(1)\\
1/10 &= 0.1
\end{align}$$
<p>Where $0.1(6)$ means $0.166666\cdots$, and has a $1$-digit recurring cycle. It can be seen that $1/7$ has a $6$-digit recurring cycle.</p><p>Find the value of $d \lt 1000$ for which $1/d$ contains the longest recurring cycle in its decimal fraction part.</p>

### Solution notes
I wanted to do this in a cool way without using long division. After a lot of effort I got it working without numba, but it is very slow in this first implementation.

This method first checks if the unit is a divisor of a power of 10. For example, $4$ is a divisor of $10^2$ and therefore $\frac{1}{4}$ can be written as $\frac{25}{100}$. In base 10, having a power of 10 as the denominator of a fraction means we have found the exact decimal representation of the original fraction. E.G. $\frac{1}{4} = \frac{25}{100} = 0.25$ These decimal representations do not have any repeating digits.

Next, we check if the unit is a divisor of a power of 10 - 1. For example, $7$ is not a divisor of $10^n$ for any $n$. But it is a divisor of $10^6-1$ as $10^6-1 = 999.999$ and $999.999/7 = 142.857$. This is very close to a power of 10, and can therefore nearly be interpreted as a decimal, however as it is not exactly a power of 10, the actual result is a repeating decimal. This means $\frac{1}{7} = \frac{142857}{999999} = 0.\overline{142857}$

If the unit is not a divisor of either of the above, but it is a divisor of $(10^n-1) \times 10^m$, this means it is a repeating fraction with leading zeros. Though I could not quickly find an example of this in the unit fractions (maybe there is none?), what makes this useful is subtracting other fractions from our fraction to turn them into this form. For example: $\frac{1}{12} = \frac{75}{900} = \frac{72}{900} + \frac{3}{900} = \frac{8}{100} + \frac{3}{900}$ In this example, we can separate the $\frac{8}{100}$, to be a separate fraction. As this has $10^n$ as denominator, it is a terminating decimal, and is preceding our repeating decimal. The remainder is a divisor of $(10^n-1) \times 10^m$, meaning this is our repeating decimal preceded by lead zeros. With this final trick, the decimal representations of all unit fractions can now be determined without long division.

I tried making a numba-compatible version of this function to make it much faster, however, this proved very complex due to the constraints on integer size in numba. So instead, I will first implement a long division method, to see how fast that is. I can then also compare the time of the long division method without numba, to determine if making an improved version of the interesting method described above is worth the effort.

In [3]:
@solution(P026, first=True, max_tests= 0, make_fast=False, warmup_args=(P026.upper_limit,))
def repeating_decimal_options(d):
    def get_unit_decimal(unit, preceding= "0.", running_decimal_places= 0):
        digit_max = 1001
        for exponent in range(1, digit_max):
            denominator = (10 ** exponent)
            # If the original denominator is a divider of a power of 10, we can rewrite the fraction as numerator/10^n. In this case, the numerator is the same as what will be behind the comma in the decimal representation.
            if denominator % unit == 0:
                return preceding, str(denominator // unit).zfill(exponent), False
            # If, instead the fraction can be written as numerator/(10^n)-1, this has nearly the same effect, and this numerator is a repeating decimal which approaches the numerator of numerator/10^n. Sometimes, we need to remove a preceding fraction before moving to the
            denominator = (denominator - 1) * 10 ** running_decimal_places
            if denominator % unit == 0:
                return preceding, str(denominator // unit).zfill(exponent), True
        for denominator_power_of_10 in range(1, digit_max):
            for number_of_nines in range(1, digit_max):
                denominator = ((10 ** number_of_nines) - 1) * 10 ** denominator_power_of_10
                if  denominator % unit == 0:
                    new_fraction = denominator // unit
                    tenth = denominator // (10 ** denominator_power_of_10)
                    bonus_fraction = new_fraction // tenth #(10 ** (number_of_nines + denominator_power_of_10 - 2))
                    next_fraction = new_fraction % tenth
                    if bonus_fraction != 0:
                        zeros = ""
                        # print(running_decimal_places)
                        if running_decimal_places == 0:
                            zeros = "0" * (denominator_power_of_10 - 1)
                        return get_unit_decimal(denominator // next_fraction, preceding + zeros + str(bonus_fraction), running_decimal_places + 1)
                    return get_unit_decimal(denominator // next_fraction, preceding, running_decimal_places + 1)

        return None, None, False
    longest_d = 0
    longest_d_length = 0
    non_repeating = 0
    for i in range(1, d):
        preceding, fraction, repeating = get_unit_decimal(i)
        if repeating:
            if len(fraction) > longest_d_length:
                longest_d_length = len(fraction)
                longest_d= i
        else:
            non_repeating += 1
    # print(non_repeating)
    #     if not fraction:
    #         print("1/{0} = not found".format(i))
    #         continue
    #     if repeating:
    #         if preceding != "0.":
    #             print("1/{0} = {1} = {2}({3}) repeating".format(i,1/i,preceding, fraction))
    #         else:
    #             print("1/{0} = {1} = 0.({2}) repeating".format(i,1/i,fraction))
    #     else:
    #         print("1/{0} = {1} = 0.{2}".format(i,1/i, fraction))

    return longest_d

In [4]:
 p.test_once("repeating_decimal_options")

983 found after a separate test in 67753.191900 ms by repeating_decimal_options (first)


After thinking about how to implement the long division method for a few seconds, I have already realised that this will probably be much faster than what I was doing before. This is owing to the fact that, though we will make many calculations to get to our answer, all calculations will be very small. For a divisor of n digits, we will never need to do calculations on any number bigger than n + 1 digits. This not only makes this method instantly numba-compatible (our divisors only go up to 999 in this problem, so the biggest integer we could potentially encounter is 9999), it also makes the calculations we have to do much faster. Furthermore, the other method would require further tinkering to ignore the actual decimals (we are not interested in the resulting decimal, after all, only in which $d$ gives the longest repeating chain). The fastest (and easiest to implement) version of this method will not calculate the decimal at all unless explicitly added.

As we are doing long division with a numerator of $1$, we will be using powers of $10$ for our calculations. For any unit, we will use the power of $10$ of the amount of digits that unit has, as this is the smallest it will divide into, with the exception of full powers of $10$. So for example, $231$ has $3$ digits, therefore, the smallest power of $10$ it will divide into is $10^3=1000$. $100$, is itself a power of $10$, so though it has $3$ digits, it will divide into $10^2$.

In [5]:
@solution(P026, make_fast= True, warmup_args=(P026.upper_limit,))
def long_division(d):
    max_length = 0
    longest_d = 0
    for i in range(1, d):
        length = 0
        # You can also calculate the digits of a number with log10, which is quickly more efficient. These values
        # will stay very low, though, so string conversion is still the faster option.
        numerator = 10 ** len(str(i))
        past_numerators = []
        if i % 10 == 0:
            numerator /= 10
        while True:
            length += 1
            numerator = numerator % i
            # Not a repeating fraction
            if numerator == 0:
                break
            if numerator in past_numerators:
                if length > max_length:
                    max_length = length
                    longest_d = i
                break
            past_numerators.append(numerator)
            while i > numerator:
                numerator *= 10
    return longest_d


In [6]:
p.test_once("long_division")

983 found after a separate test in 4.809100 ms by long_division


Because I am stubborn, I want to take another look at my first solution which, in my opinion, is much cooler than simple long division. I don't know if it can become a viable alternative for this problem, but I do want to improve it. If I improve it enough, it may at least become interesting enough to be the solution to a new problem I could create.

With that in mind, the main improvements will lie in decreasing the size of the for loops. The first for loop currently runs the arbitrarily chosen limit of $1000$ times. The second nested loops run $1000^2$ times, meaning we have a total of $1.001.000$ potential iterations. To reduce this number, let's investigate what we can accomplish by looking at the factors of the unit denominators.

If the prime factorisation of a unit denominator is of the form $2^n \times 5^m$, this unit fraction terminates. This is because it can be written in the form $\frac{x}{10^y}$ which is a terminating decimal number.

If the prime factorisation of a unit denominator does not contain a $2$ or a $5$, it is a repeating fraction, without any preceding digits. In other words, $\frac{1}{n} = 0.\overline{d_1, d_2 ... d_k}$ for any $n$ if $n\bmod 2 \neq 0 \wedge n\bmod 5 \neq 0$. This also means this fraction can be written in the form $\frac{x}{10^y-1}$ where $x$ are the repeating digits.

If the prime factorisation of a unit denominator contains a $2$ and/or a $5$ as well as other primes, it is a repeating fraction, but preceded by other digits. E.G. $\frac{1}{6} = 0.1\overline{6}$ or $\frac{1}{30} = 0.0\overline{3}$. If the prime factorisation contains both a $2$ and a $5$, we need not calculate any further, as this can be reduced to a fraction of which we have already checked the number of repeating digits. E.G. $\frac{1}{30} = \frac{1}{10} * \frac{1}{3}$. The repeating digits are not more than the previously calculated fraction.

If the prime factorisation of $n$ has at least one $2$ or $5$, but none of the other, as well as other prime factors it can also be re-written as:
 $\frac{1}{n} = \frac{1}{2^a} \times \frac{1}{n/2^a} \wedge \frac{1}{5^a} \times \frac{1}{n/5^a}$.

This is a terminating fraction multiplied by a previously calculated repeating fraction. This does not increase its number of repeating digits, so these cases also need not be checked.

In [7]:
@solution(P026, make_fast= False, warmup_args=(P026.upper_limit,))
def improved_decimal_options(d):
    max_length = 0
    longest_d = 0
    for i in range(2, d):
        if i % 10 == 0:
            # This i is 10^n * its other prime factors. This means it has preceding digits, but only zeros.
            # It also means, the repeating part of the fraction is the same as that of i/10^n and does not need to be calculated
            continue
        divisor = 2 if i % 2 == 0 else 5 if i % 5 == 0 else 0
        if divisor:
            remainder = i
            while remainder % divisor == 0:
                remainder = remainder // divisor
            if remainder == 1:
                # This i is only divisible by 2 or 5 and is, therefore, not repeating
                continue
            # i is divisible by 2 or 5, not both, and also has other prime factors. This number of repeating digits
            # was already calculated earlier, so we can continue.
            continue

        power_of_10 = 10
        repeating_digits = 1
        while True:
            if (power_of_10 -1) % i == 0:
                if repeating_digits > max_length:
                    max_length = repeating_digits
                    longest_d = i
                # print(f"{i} = {1/i}: {repeating_digits} repeating digit(s)")
                break
            # print(f"{i} = {1/i} = preceding non-zeros ")
            power_of_10 *= 10
            repeating_digits += 1
    return longest_d


In [8]:
p.test_once("improved_decimal_options")

983 found after a separate test in 13.046300 ms by improved_decimal_options


The main bottleneck in the decimal options method is still the final calculation, which is based on using big integers. Numba cannot handle these. Instead, combining the two methods discussed so far results in the fastest function yet. Replacing the list with a boolean array makes the long division tracking much faster, and running all of the previously discussed checks prevents unnecessary calculations. This results in a function for long division which only calculates the required fractions and no others.

The final addition to this function is based on the fact that with all of the previously discussed reasoning steps, we can compact the logic to prevent unnecessary calculations to a very simple check: if i is divisable by 2 or 5, we need not check it for various reasons.

In [9]:
@solution(P026, best= True, make_fast= True, warmup_args=(P026.upper_limit,))
def long_division_decimal_options(d):
    max_length = 0
    longest_d = 0
    for i in range(2, d):
        if i % 2 == 0 or i % 5 == 0:
            continue

        length = 0
        numerator = 10 ** len(str(i))
        past_numerators = np.zeros(d, dtype=np.bool_)
        while True:
            length += 1
            numerator = numerator % i
            if past_numerators[numerator]:
                if length > max_length:
                    max_length = length
                    longest_d = i
                break
            past_numerators[numerator] = True
            while i > numerator:
                numerator *= 10
    return longest_d


In [10]:
p.test_all()

983 found after 1000 tests in 13.134905 ms by improved_decimal_options
983 found after 1000 tests in 4.649961 ms by long_division
983 found after 1000 tests in 0.399396 ms by long_division_decimal_options (best)
983 found after a separate test in 67753.191900 ms by repeating_decimal_options (first)
